In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn.functional as F

# Configuración del dispositivo
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Dataset y DataLoaders (CIFAR-10)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Limitamos el dataset para agilizar pruebas
trainset_limited = torch.utils.data.Subset(trainset, range(2000))
testset_limited = torch.utils.data.Subset(testset, range(500))

batch_size = 16
trainloader = torch.utils.data.DataLoader(trainset_limited, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset_limited, batch_size=batch_size, shuffle=False, num_workers=2)

# CARGAR EL MODELO TEACHER DESDE SU ARCHIVO .PTH
# Definimos la arquitectura base del Teacher (asegúrate de que coincida con la que guardaste)
teacher_model = models.resnet18(weights=None)
teacher_model.fc = nn.Linear(teacher_model.fc.in_features, 10) # 10 clases para CIFAR-10

# Ruta a tu archivo .pth del Teacher
teacher_path = 'cifar_net_resnet.pth'  # <--- Reemplaza con la ruta de tu pth del teacher

try:
    # Si guardaste solo el state_dict():
    teacher_model.load_state_dict(torch.load(teacher_path, map_location=device))
    print(f"Pesos del Teacher cargados exitosamente desde '{teacher_path}'")
except Exception as e:
    print(f"Error al cargar el Teacher (intenta verificar si guardaste el modelo completo o solo el state_dict): {e}")

teacher_model.to(device)
teacher_model.eval()  # El profesor se queda congelado en modo evaluación

# Congelar los parámetros del Teacher para que no se actualicen
for param in teacher_model.parameters():
    param.requires_grad = False

# student
class StudentNet(nn.Module):
    def __init__(self):
        super(StudentNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        # Nota: Si tus imágenes pasan por Resize((224, 224)), asegúrate de calcular bien 
        # las dimensiones de entrada para Linear si difieren de las originales de 32x32.
        # Aquí mantenemos la estructura original del notebook 1:
        self.fc1 = nn.Linear(16 * 53 * 53, 120)  # Ajustado de acuerdo a 224x224 después de conv/pool
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # Aplanado dinámico adaptado al batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

student_model = StudentNet().to(device)

# Student pth
student_path = '/home/fpineda/python_project/ia_course/PUCP_DL/cifar_net.pth'  # <--- Reemplaza si tienes un pth para el student, o coméntalo si empieza de cero
try:
    student_model.load_state_dict(torch.load(student_path, map_location=device))
    print(f"Pesos del Student cargados exitosamente desde '{student_path}'")
except:
    print("No se encontró o no se cargó un .pth previo para el Student. Se entrenará desde cero con destilación.")


# Función de pérdida para Knowledge Distillation
def distillation_loss(outputs, labels, teacher_outputs, alpha=0.5, temperature=4.0):
    ce_loss = nn.CrossEntropyLoss()(outputs, labels)
    kd_loss = nn.KLDivLoss(reduction='batchmean')(
        F.log_softmax(outputs / temperature, dim=1),
        F.softmax(teacher_outputs / temperature, dim=1)
    ) * (temperature ** 2)
    
    loss = (1.0 - alpha) * ce_loss + alpha * kd_loss
    return loss

# Optimizador para el Student
optimizer = optim.Adam(student_model.parameters(), lr=0.001)

#  Entrenamiento por Destilación
epochs = 3
print("Iniciando entrenamiento por Destilación de Conocimiento hacia ResNet18...")

for epoch in range(epochs):
    student_model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()

        # Predicciones del estudiante
        outputs = student_model(inputs)
        
        # Predicciones del profesor (sin gradientes)
        with torch.no_grad():
            teacher_outputs = teacher_model(inputs)

        # Calcular pérdida combinada
        loss = distillation_loss(outputs, labels, teacher_outputs, alpha=0.7, temperature=3.0)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calcular métricas de entrenamiento
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_accuracy = 100 * correct_train / total_train
    epoch_loss = running_loss / len(trainloader)

    # --- FASE DE EVALUACIÓN (TEST ACCURACY) ---
    student_model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = student_model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_accuracy = 100 * correct_test / total_test

    print(f"Época [{epoch + 1}/{epochs}] ---> Loss: {epoch_loss:.4f} | Acc (Train): {train_accuracy:.2f}% | Acc (Test): {test_accuracy:.2f}%")

print("¡ destilación finalizados!")

# Guardar el nuevo modelo Student destilado
torch.save(student_model.state_dict(), 'student_distilled_net.pth')
print("Modelo student destilado guardado como 'student_distilled_net.pth'")

Usando dispositivo: cuda:0
Pesos del Teacher cargados exitosamente desde 'cifar_net_resnet.pth'
No se encontró o no se cargó un .pth previo para el Student. Se entrenará desde cero con destilación.
Iniciando entrenamiento por Destilación de Conocimiento hacia ResNet18...
Época [1/3] ---> Loss: 5.6406 | Acc (Train): 15.85% | Acc (Test): 20.20%
Época [2/3] ---> Loss: 5.3167 | Acc (Train): 19.25% | Acc (Test): 14.80%
Época [3/3] ---> Loss: 5.2089 | Acc (Train): 20.80% | Acc (Test): 21.40%
¡ destilación finalizados!
Modelo student destilado guardado como 'student_distilled_net.pth'
